[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-08-ValueFactor-ZScore <<](./QC-Py-Cloud-08-ValueFactor-ZScore.ipynb)

# Option Wheel — Le paradoxe du win-rate eleve

**Source** : Projet etudiant ESGF 2026 (QC `31846074`), anonymise et adapte.
**Type** : `doc cloud` — resultats obtenus sur [QuantConnect Cloud](https://www.quantconnect.com/lab).

---

## Objectifs d'apprentissage

1. Comprendre la **strategie Wheel** : cycle put nu -> assignment -> covered call -> cession.
2. Distinguer **win-rate eleve** et **esperance de gain positive** (le piege fondamental).
3. Analyser le **risque de queue** : pourquoi une strategie qui gagne 90% du temps peut etre perdante.
4. Interpreter un drawdown catastrophique (-100%+) et son origine (concentration, levier implicite).

**Prerequis** : Options de base (call/put, strike, premium), QC-Py-06-Options-Trading.

**Duree estimee** : 35 min

## 1. Concept : La strategie Wheel

Le **Wheel** (roue) est une strategie d'options income populaire chez les investisseurs particuliers. Elle vise a generer un revenu regulier par la vente d'options, en faisant "tourner" un cycle.

### Le cycle en 4 etapes

```
  (1) Vendre PUT nu (cash-secured)          (3) Vendre COVERED CALL
      sur un sous-jacent que l'on                 sur les actions detenues
      est pret a acheter                          pour generer un revenu
            |                                          ^
            v                                          |
  (2) Si assigne : acheter 100 actions        (4) Si call exerce : vendre
      au strike (devenir actionnaire)            les actions -> retour a (1)
```

### La promesse

- **Revenu regulier** : on encaisse les premiums (time decay theta).
- **Win-rate eleve** : la plupart des options expirent sans valeur (~70-85% si OTM).
- **Recuperation** : si assigne, on garde les actions et on vend des calls.

### La realite cachee

> **Le piege** : un win-rate de 85% ne veut rien dire si les 15% de pertes sont 10x plus grandes que les gains. La vente d'options a un **profil de payoff asymetrique** : petits gains frequents, pertes rares mais catastrophiques.

## 2. Le paradoxe du win-rate

### Esperance mathematique

$$E[G] = p_{win} \times gain_{moyen} - p_{loss} \times perte_{moyenne}$$

Exemple typique d'un put OTM :

| Scenario | Probabilite | P&L par contrat |
|----------|-------------|------------------|
| Option expire sans valeur | 85% | +200$ (premium garde) |
| Option assignee (crash) | 15% | -1 800$ (achat a strike > marche) |

$$E[G] = 0.85 \times 200 - 0.15 \times 1800 = 170 - 270 = -60$$$

**Esperance NEGATIVE** malgre un win-rate de 85% ! C'est le paradoxe fondamental.

### Pourquoi c'est seduisant psychologiquement

1. **Renforcement positif frequent** : on encaisse des premiums presque chaque semaine.
2. **Recence bias** : les grosses pertes sont rares, donc oubliees.
3. **Illusion de competence** : "ca marche", donc on augmente la taille.
4. **Vente de volatilite** : on parie que le marche sera calme — jusqu'a ce qu'il ne le soit plus.

### Exercice 1 : Calculer l'esperance de gain

Implementez la fonction `expected_value` qui calcule l'esperance d'une vente de put, etant donnes la probabilite d'assignment, la premium et la perte moyenne si assigne.

In [1]:
# Exercice 1 : Calculer l'esperance de gain d'une vente de put
# TODO etudiant : implementer expected_value(p_assignment, premium, avg_loss) -> float
# Indice : E[G] = (1 - p_assignment) * premium - p_assignment * avg_loss
def expected_value(p_assignment, premium, avg_loss):
    return 0.0  # TODO etudiant : calculer l'esperance


## 3. Le code QC Cloud

La strategie `WheelTechSimple` (projet QC `31846074`) implemente le cycle sur 5 actions tech (NVDA, ORCL, CSCO, AMD, QCOM) :

```python
class WheelTechSimple(QCAlgorithm):
    def Initialize(self):
        self.SetStartDate(2020, 1, 1)
        self.SetEndDate(2024, 12, 31)
        self.SetCash(50000)
        self.SetBenchmark("VGT")  # Tech ETF
        
        # Parametres du wheel
        self.days_to_expiry = 30    # DTE cible
        self.otm_threshold = 0.05   # 5% OTM minimum
        self.max_exposure = 0.20    # 20% du capital par trade
        
        # Univers : 5 actions tech volatiles
        self.eq_names = ['NVDA', 'ORCL', 'CSCO', 'AMD', 'QCOM']
```

### Logique du cycle

- `OnData` verifie chaque action : si pas de position -> vendre put ; si on a les actions -> vendre covered call.
- `_sell_put` : calcule le strike OTM, trouve le contrat, vend la quantite (max 20% cash).
- `_sell_covered_call` : vend des calls couverts sur les actions detenues.
- `_find_option` : filtre la chaine d'options par DTE et strike.

## 4. Resultats QC Cloud — Le piege revele

**Projet QC Cloud** : `31846074` (VGT tech wheel, 2020-2024).

| Metrique | Valeur | Benchmark VGT (~) | Interpretation |
|----------|--------|--------------------|----------------|
| Sharpe | -0.51 | ~0.70 | **Nettement negatif** |
| CAGR | 0% | ~18% | Aucun rendement |
| MaxDD | -103.5% | ~-30% | **Catastrophique** (perte virtuelle totale) |
| PSR | 0.004% | > 50% | Signal inexistant |
| Tradeable days | 1258 | — | 5 ans de donnees |

### Diagnostic

**Sharpe -0.51** : la strategie detrui de la valeur. La vente de premiums ne compense pas les pertes d'assignment.

**MaxDD -103.5%** : anormal. Un drawdown theorique > 100% indique un **levier implicite** ou une **concentration extreme** : si plusieurs puts sont assignes simultanement pendant un crash (ex: COVID mars 2020, bear market 2022), le capital ne suffit pas a couvrir les achats forces au-dessus du marche.

**CAGR 0% avec Sharpe negatif** : le cash reste immobilise en collateral (cash-secured puts), generant peu de rendement, tandis que les assignments successifs erodent le capital.

> **Lecon** : le wheel sur actions tech volatiles pendant une periode de crash (2020 COVID + 2022 bear market) illustre exactement le paradoxe. Les premiums etaient petits, les assignments frequents et couteux.

### Exercice 2 : Identifier les scenarii defavorables

Le wheel perd de l'argent dans des conditions specifiques. Identifiez les 3 principaux scenarii de perte et proposez une mitigation pour chacun.

In [2]:
# Exercice 2 : Identifier les scenarii de perte du wheel
# TODO etudiant : pour chaque scenario, decrire le mecanisme de perte + 1 mitigation
loss_scenarios = {
    "crash_concurrent": None,    # TODO etudiant : description + mitigation
    "action_devaluee": None,     # ex: action assignee qui continue de chuter
    "volatilite_implose": None,  # IV crush apres un evenement
}
# Indice : la diversification et le position sizing sont les premieres mitigations
print("Exercice a completer")

Exercice a completer


## 5. Lecons de gestion du risque

### 1. Le win-rate n'est pas une metrique de performance

Une strategie a 90% de win-rate peut etre perdante. La vraie question est l'**esperance** et le **rapport gain/perte**. Un Sharpe positif est plus informatif qu'un win-rate eleve.

### 2. Le risque de queue est real

La vente d'options vend de l'assurance. Comme un assureur, on encaisse des petites primes jusqu'au sinistre — puis on paie gros. Les evenements extremes (COVID, crash, faillite) surviennent plus souvent que le modele gaussien ne le predit (fat tails).

### 3. La concentration tue

Le wheel sur 5 actions tech (NVDA, AMD, ...) concentre le risque sectoriel. Un crash tech = assignments multiples simultanes = perte disproportionnee. Diversifier sur 20-50 sous-jacents non correlates reduit ce risque.

### 4. Le position sizing est critique

Vouloir "maximiser le revenu" pousse a vendre trop de contrats. La regle prudente : ne jamais risquer plus de 1-2% du capital par trade, et garder 50%+ de cash disponible pour les assignments.

### 5. Le regime de volatilite compte

Le wheel performe mieux en marche stable/haussier avec volatilite moderee. En bear market ou haute volatilite, les puts sont assignes et les covered calls limitent la recuperation. Adapter la taille au regime VIX.

### Exercice 3 : Concevoir un wheel defensif

Proposez 3 modifications pour transformer ce wheel agressif en une strategie plus defensive, en justifiant l'impact attendu sur le risque.

In [3]:
# Exercice 3 : Concevoir un wheel defensif
# TODO etudiant : 3 modifications avec justification de l'impact risque
defensive_modifications = {
    "diversification": None,  # TODO etudiant : ex: 20+ sous-jacents non correlates
    "position_sizing": None,  # ex: max 1% capital par trade
    "regime_filter": None,    # ex: desactiver si VIX > 30
}
# Indice : chaque modification reduit le revenu mais protege contre le risque de queue
print("Exercice a completer")

Exercice a completer


## Conclusion : Le wheel n'est pas un revenu gratuit

### Points cles

1. **Le wheel vend de l'assurance** : revenu regulier en echange d'un risque de queue asymetrique.

2. **Win-rate != esperance** : 85% de reussite peut cacher une esperance negative si les pertes sont 10x les gains.

3. **La concentration sectorielle amplifie le risque** : le wheel sur tech pendant 2020-2022 a expose a des assignments multiples.

4. **Le drawdown > 100% revele un levier implicite** : le collateral cash ne suffit pas quand plusieurs puts sont assignes simultanement.

5. **Le regime de volatilite est decisif** : adapter la taille et la frequence au VIX, et non viser un revenu fixe.

### Pour aller plus loin

- **Backtester avec regime filter** : desactiver la vente de puts quand VIX > 30 ou trend SPY < 200-MA.
- **Comparer a un buy-and-hold** : le wheel doit battre le benchmark APRES couts et taxes, pas juste en premiums brutes.
- **Diversifier l'univers** : 20-50 sous-jacents non correlates vs 5 actions tech.

---

[<< Sommaire QC](../README.md) | [Precedent : QC-Py-Cloud-08-ValueFactor-ZScore <<](./QC-Py-Cloud-08-ValueFactor-ZScore.ipynb)